# Sentiment Analysis
- Using Machine learning 
- Using Deep Learning (LSTM)
- Using BERT


#### Student: Eric Michel
September 13, 2024


# Sentiment analysis in NLP

References:

https://vadersentiment.readthedocs.io/en/latest/#:~:text=VADER%20(Valence%20Aware%20Dictionary%20and%20sEntiment%20Reasoner)%20is%20a%20lexicon


https://github.com/cjhutto/vaderSentiment/tree/master/vaderSentiment

In [69]:
# !pip install vaderSentiment

# !pip install pandas numpy scikit-learn nltk

In [70]:
#Sentiment analysis in NLP is a technique to determine the emotional tone behind the body of the text
# 1. Rule Based System
# 2. Transformer based system

In [71]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
import nltk
from nltk.corpus import stopwords
import string

nltk.download('stopwords')


import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout


[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/oysterable/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


# Basic Rule Based System

In [72]:
#Basic Rule Based System

#Create and supply dictionaries containing positive and negative words
positiveWords = ["good","happy","excellent","great","positive","fortunate","correct"]
negativeWords = ["bad","sad","poor","negative","unfortunate","wrong","inferior","terrible"]


In [73]:
#Simple Analyser function
def ruleBasedSimpleTextSentimentAnalyser(text):
  #Normalization
  text = text.lower()

  #Initialize sentiment score
  positiveCount=0
  negativeCount=0

  #Tokenize the text into words
  words = text.split()

  #Check each word with my dictionary to identify number of positive and negative words
  for word in words:
    if word in positiveWords:
      positiveCount += 1
    elif word in negativeWords:
      negativeCount +=1

  #Determine sentiment
  if positiveCount > negativeCount:
    return "Positive"
  elif negativeCount > positiveCount:
    return "Negative"
  else:
    return "Neutral"

In [74]:
ruleBasedSimpleTextSentimentAnalyser("This product is great and works perfectly")

'Positive'

# VADER (Valence Aware Dictionary and sEntiment Reasoner)

In [75]:

# VADER uses rulebased approach with pre-defined lexicon which maps words to their sentiment intensity scores
# It considers the following:
# 1. Exclamation points
# 2. Emphasize on Capital Letter
# 3. Negation words
# 4. Level of sentiment (Degree Modifiers) -- very, extremely, super-excited

In [76]:
#VADER Scoring System
# VADER assigns each word in a text data a score between -4 to +4
# Positive words have score close to +4
# Negative words have score close to -4
# Neutral words have score close to 0

# VADER Metric
# 1. Positive Score : Proportion of the text with positive sentiment
# 2. Negative Score : Proportion of text with negative sentiment
# 3. Neutral Score : Proportion of text with neutral sentiment
# 4. Compound Score : Overall sentiment score; ranging from -1(most negative) to +1(most positive)

In [77]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

analyser = SentimentIntensityAnalyzer()

analyser.polarity_scores("I love this product! I hate this product")

{'neg': 0.261, 'neu': 0.423, 'pos': 0.317, 'compound': 0.2003}

In [117]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

def sentimentClassification(text):
  
  analyser = SentimentIntensityAnalyzer()
  scores = analyser.polarity_scores(text)
  compoundScore = scores['compound']

  if compoundScore >= 0.05:
    return "positive"
  elif compoundScore <= -0.05:
    return "negative"
  else:
    return "neutral"

In [79]:
sentimentClassification(("I love this product! It's amazing"))

'Positive'

Task

In [80]:
# yelp dataset (Day3 assignment folder)
# Using Vader create a new column named sentiment and find sentiment for each text/review and store the same in the sentiment column
# Create a ML model for performing sentimental analysis

In [81]:
import pandas as pd

In [155]:

data = pd.read_csv('./Datasets/yelp.csv')

data.columns

Index(['business_id', 'date', 'review_id', 'stars', 'text', 'type', 'user_id',
       'cool', 'useful', 'funny', 'sentiment'],
      dtype='object')

In [150]:
# Drop rows where 'text' is null
data = data.dropna(subset=['text'])
data.dropna(inplace=True)  # Drop missing values

# Ensure all values in the 'text' column are strings
data['text'] = data['text'].astype(str)

In [151]:
data[['text']]

,text
0,My wife took me here on my birthday for breakf...
1,I have no idea why some people give bad review...
2,love the gyro plate. Rice is so good and I als...
3,"Rosie, Dakota, and I LOVE Chaparral Dog Park!!..."
4,General Manager Scott Petello is a good egg!!!...
...,...
9995,First visit...Had lunch here today - used my G...
9996,Should be called house of deliciousness!\n\nI ...
9997,I recently visited Olive and Ivy for business ...
9998,My nephew just moved to Scottsdale recently so...


In [152]:
# # Find the sentiment for each text/review
# data['sentiment'] = data['text'].apply(sentimentClassification)
## Save the dataset
# data.to_csv('./Datasets/yelp.csv', index=False)
# data.head()

,business_id,date,review_id,stars,text,type,user_id,cool,useful,funny,sentiment
0,9yKzy9PApeiPPOUJEtnvkg,2011-01-26,fWKvX83p0-ka4JS3dc6E5A,5,My wife took me here on my birthday for breakf...,review,rLtl8ZkDX5vH5nAx9C3q5Q,2,5,0,positive
1,ZRJwVLyzEJq1VAihDhYiow,2011-07-27,IjZ33sJrzXqU-0X6U8NwyA,5,I have no idea why some people give bad review...,review,0a2KyEL0d3Yb1V6aivbIuQ,0,0,0,positive
2,6oRAC4uyJCsJl1X0WZpVSA,2012-06-14,IESLBzqUCLdSzSqm0eCSxQ,4,love the gyro plate. Rice is so good and I als...,review,0hT2KtfLiobPvh6cDC8JQg,0,1,0,positive
3,_1QQZuf4zZOyFCvXc0o6Vg,2010-05-27,G-WvGaISbqqaMHlNnByodA,5,"Rosie, Dakota, and I LOVE Chaparral Dog Park!!...",review,uZetl9T0NcROGOyFfughhg,1,2,0,positive
4,6ozycU1RpktNG2-1BroVtw,2012-01-05,1uJFq2r5QfJG_6ExMRCaGw,5,General Manager Scott Petello is a good egg!!!...,review,vYmM4KTsC8ZfQBg-j5MWkw,0,0,0,positive


In [156]:
data.sentiment.value_counts()

sentiment
positive    8950
negative     921
neutral      129
Name: count, dtype: int64

# Create a ML model for sentiment analysis

Preprocess Data

In [123]:
stop_words = set(stopwords.words('english'))

def preprocess(text):
    text = text.lower()  # Convert to lowercase
    text = ''.join([char for char in text if char not in string.punctuation])  # Remove punctuation
    words = text.split()  # Split into words
    words = [word for word in words if word not in stop_words]  # Remove stopwords
    return ' '.join(words)

# Apply preprocessing
data['processed_text'] = data['text'].apply(preprocess)

In [124]:
X = data['processed_text']
y = data['sentiment']
y

0       positive
1       positive
2       positive
3       positive
4       positive
          ...   
9995    positive
9996    positive
9997    positive
9998    negative
9999    positive
Name: sentiment, Length: 10000, dtype: object

In [1]:
X[-2]

NameError: name 'X' is not defined

Train-Test Split

In [ ]:

# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert Text to Numeric Vectors
We’ll use TF-IDF to convert the text data into numeric features.

In [ ]:
# Convert text to TF-IDF vectors
vectorizer = TfidfVectorizer(max_features=5000)  # Use top 5000 words as features

X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)

Train Logistic Regression Model
We’ll use logistic regression, try other classifiers like SVM or Naive Bayes.

In [127]:
# Train a Logistic Regression classifier
model = LogisticRegression()
## try other classifiers like SVM or Naive Bayes.

model.fit(X_train_tfidf, y_train)

# Predict on the test set
y_pred = model.predict(X_test_tfidf)


In [128]:
# Calculate accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.4f}')

# Detailed classification report
print(classification_report(y_test, y_pred))


Accuracy: 0.9095
              precision    recall  f1-score   support

    negative       0.88      0.16      0.28       177
     neutral       0.00      0.00      0.00        30
    positive       0.91      1.00      0.95      1793

    accuracy                           0.91      2000
   macro avg       0.60      0.39      0.41      2000
weighted avg       0.89      0.91      0.88      2000



/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))
/opt/anaconda3/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1344: UndefinedMetricWarning: Precision and F-score are ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, msg_start, len(result))


Save the model

In [129]:
import pickle

# Save the model and vectorizer
with open('sentiment_model.pkl', 'wb') as model_file:
    pickle.dump(model, model_file)
with open('tfidf_vectorizer.pkl', 'wb') as vectorizer_file:
    pickle.dump(vectorizer, vectorizer_file)


Make inferences in the model

In [130]:

with open('sentiment_model.pkl', 'rb') as model_file:
    model = pickle.load(model_file)


def predict_sentiment(new_text):
    processed_text = preprocess(new_text)
    text_tfidf = vectorizer.transform([processed_text])
    prediction = model.predict(text_tfidf)
    return prediction[0]

print(predict_sentiment("The movie was fantastic!"))
print(predict_sentiment("I didn't enjoy the film."))


positive
positive


# Using Deep Learning for Sentiment Analysis

### Tokenize the Text Data

We'll tokenize the text and convert it into sequences of numbers. 

Deep learning models require the text to be converted into numeric data.

In [142]:
# Tokenize the text
tokenizer = Tokenizer(num_words=10000)  # Use the top 10,000 words
tokenizer.fit_on_texts(data['processed_text'])

# Convert text into sequences
sequences = tokenizer.texts_to_sequences(data['processed_text'])

# Pad sequences to ensure uniform input size
max_sequence_length = 800  # Pad/truncate to 100 words per text
X = pad_sequences(sequences, maxlen=max_sequence_length)

# Encode sentiment labels as categorical values for 3 classes (negative, neutral, positive)
# For example, 0 = negative, 1 = neutral, 2 = positive
label_mapping = {'negative': 0, 'neutral': 1, 'positive': 2}
data['sentiment_label'] = data['sentiment'].map(label_mapping)

# One-hot encoding of labels
y = pd.get_dummies(data['sentiment_label']).values
data['sentiment']

0       positive
1       positive
2       positive
3       positive
4       positive
          ...   
9995    positive
9996    positive
9997    positive
9998    negative
9999    positive
Name: sentiment, Length: 10000, dtype: object

In [143]:
# Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


Build the Deep Learning Model

We'll build an LSTM model. You can adjust the number of layers, units, and dropout rates to tune the model’s performance.

In [144]:
# Define the LSTM model
model = Sequential()

# Embedding layer to learn word vectors
model.add(Embedding(input_dim=10000, output_dim=128, input_length=max_sequence_length))

# LSTM layer
model.add(LSTM(units=128, return_sequences=False))

# Dropout to avoid overfitting
model.add(Dropout(0.5))

# Fully connected layer with softmax for binary classification
model.add(Dense(3, activation='softmax'))

# Compile the model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

# Summary of the model
model.summary()


/opt/anaconda3/lib/python3.11/site-packages/keras/src/layers/core/embedding.py:90: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_4 (LSTM)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

Train the Model
Train the LSTM model on the training data.

In [146]:
# Train the model
history = model.fit(X_train, y_train, epochs=10, batch_size=64, validation_split=0.2)


Epoch 1/10
100/100 ━━━━━━━━━━━━━━━━━━━━ 57s 569ms/step - accuracy: 0.8954 - loss: 0.3382 - val_accuracy: 0.9044 - val_loss: 0.2797
Epoch 2/10
 62/100 ━━━━━━━━━━━━━━━━━━━━ 20s 538ms/step - accuracy: 0.9201 - loss: 0.2084

KeyboardInterrupt: 

In [135]:
# Evaluate the model on the test set
test_loss, test_acc = model.evaluate(X_test, y_test)
print(f'Test accuracy: {test_acc:.4f}')


63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.9094 - loss: 0.4359
Test accuracy: 0.9070


In [136]:
# Save the model
model.save('sentiment_lstm_model.h5')

# To load the model later
# model = tf.keras.models.load_model('sentiment_lstm_model.h5')


In [148]:
def predict_sentiment(new_text):
    processed_text = preprocess(new_text)
    seq = tokenizer.texts_to_sequences([processed_text])
    padded_seq = pad_sequences(seq, maxlen=max_sequence_length)
    prediction = model.predict(padded_seq)
    sentiment = np.argmax(prediction)  # Get the index of the highest probability
    label_mapping_reverse = {0: 'negative', 1: 'neutral', 2: 'positive'}
    return label_mapping_reverse[sentiment]

print(predict_sentiment("The movie was fantastic!"))  # Expected output: 'positive'
print(predict_sentiment("It was okay, not the best but not the worst either."))  # Expected output: 'neutral'
print(predict_sentiment("The movie was bad."))  # Expected output: 'negative'


# "I love this movie. It's absolutely fantastic!",
# "This was the worst movie I've ever seen. Total waste of time.",
# "It was okay, not the best but not the worst either.",
# "The acting was bad, but the plot was good."

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 133ms/step
positive
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 61ms/step
neutral
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step
neutral


# Sentiment Analysis Using BERT

In [ ]:
from datasets import Dataset

# Remove unnecessary columns
clean_dataset = data[['text','sentiment_label']]

dataset = Dataset.from_pandas(clean_dataset)

# Peek at the dataset to ensure it's loaded correctly
print(dataset)

In [ ]:
from transformers import BertTokenizer

# Load the pre-trained BERT tokenizer
tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')

# Tokenize function to apply to each example
def tokenize_function(examples):
    return tokenizer(examples['text'], padding='max_length', truncation=True)

# Apply the tokenizer to the dataset
tokenized_dataset = dataset.map(tokenize_function, batched=True)

# Rename the sentiment_label column to 'labels' since this is mandatory for PyTorch
tokenized_dataset = tokenized_dataset.rename_column('sentiment_label', 'labels')

# Set the dataset format to PyTorch tensors
tokenized_dataset.set_format('torch')

# Split the dataset into training and testing sets (80% train, 20% test)
train_test_split = tokenized_dataset.train_test_split(test_size=0.2)
train_dataset = train_test_split['train']
test_dataset = train_test_split['test']

print(train_dataset)
print(test_dataset)

train_dataset['labels']

In [ ]:
from transformers import BertForSequenceClassification

# Load pre-trained BERT for sequence classification (with 3 sentiment labels)
model = BertForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=3)


In [ ]:
from transformers import Trainer, TrainingArguments


num_epocs = 10

# Define the training arguments
training_args = TrainingArguments(
    output_dir='./results',          # output directory to save model
    evaluation_strategy="epoch",     # evaluate each epoch
    per_device_train_batch_size=16,  # batch size for training
    per_device_eval_batch_size=16,   # batch size for evaluation
    num_train_epochs=num_epocs,      # number of epochs
    weight_decay=0.01,               # strength of weight decay
    logging_dir='./logs',            # directory for logs
    logging_steps=10,
)

# Initialize the trainer
trainer = Trainer(
    model=model,                     # the pre-trained BERT model
    args=training_args,              # training arguments
    train_dataset=train_dataset,     # training dataset
    eval_dataset=test_dataset        # evaluation dataset
)


In [ ]:
# Train the model
trainer.train()

In [ ]:
# Evaluate the model
results = trainer.evaluate()
print(results)

In [ ]:
# Save the model and tokenizer
model.save_pretrained('./fine_tuned_model')
tokenizer.save_pretrained('./fine_tuned_model')

In [ ]:
from transformers import BertTokenizer, BertForSequenceClassification
import torch


# Load the saved tokenizer
tokenizer = BertTokenizer.from_pretrained('./fine_tuned_model')

# Load the saved model
model = BertForSequenceClassification.from_pretrained('./fine_tuned_model')

# Example text to classify sentiment
text = "The movie was amazing and I loved it!"

# Tokenize the input text (same as how it was done during training)
inputs = tokenizer(text, return_tensors='pt', padding='max_length', truncation=True)

In [ ]:
# Perform inference (get the logits)
outputs = model(**inputs)

# Extract the predicted label (index of the maximum value in logits)
predictions = torch.argmax(outputs.logits, dim=1)

# Map the prediction to the actual label (e.g., positive, negative, neutral)
label_map = {0: 'Negative', 1: 'Neutral', 2: 'Positive'}  # Adjust according to your label mapping
predicted_label = label_map[predictions.item()]

print(f"Predicted sentiment: {predicted_label}")
